In [66]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [67]:
df = pd.read_csv(r'C:\Users\Kato\Version3.12\bank-marketing-analysis\data\bank-full.csv', sep=';')
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [68]:
df['y'].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [70]:
df.shape

(45211, 17)

In [71]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

In [72]:
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [73]:
numerical_cols = ['age', 'balance', 'campaign', 'previous', 'day', 'was_previously_contacted', 'days_since_prev_contact', 'month_sin', 'month_cos']
nominal_categorical = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'poutcome', 'education']
#calendar = ['month'] converted to month_sin and month_cos
#special = ['pdays'] performed feature engineering on this
categorical_cols = [k for k in df.columns if k not in numerical_cols and k != 'y']

In [74]:
for col in df.columns:
    print(f'--- {col} ---')
    print(df[col].nunique())
    print('----------------')
    print(df[col].value_counts().head(5))

--- age ---
77
----------------
age
32    2085
31    1996
33    1972
34    1930
35    1894
Name: count, dtype: int64
--- job ---
12
----------------
job
blue-collar    9732
management     9458
technician     7597
admin.         5171
services       4154
Name: count, dtype: int64
--- marital ---
3
----------------
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64
--- education ---
4
----------------
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64
--- default ---
2
----------------
default
no     44396
yes      815
Name: count, dtype: int64
--- balance ---
7168
----------------
balance
0    3514
1     195
2     156
4     139
3     134
Name: count, dtype: int64
--- housing ---
2
----------------
housing
yes    25130
no     20081
Name: count, dtype: int64
--- loan ---
2
----------------
loan
no     37967
yes     7244
Name: count, dtype: int64
--- contact ---
3
----------------
contact
cellu

In [75]:
print(df['day'].describe())

count    45211.000000
mean        15.806419
std          8.322476
min          1.000000
25%          8.000000
50%         16.000000
75%         21.000000
max         31.000000
Name: day, dtype: float64


In [76]:
df['was_previously_contacted'] = (df['pdays'] != -1).astype(int)
df['days_since_prev_contact'] = df['pdays'].replace(-1, np.nan)
df.drop('pdays', axis=1, inplace=True)
df.tail(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,previous,poutcome,y,was_previously_contacted,days_since_prev_contact
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,0,unknown,yes,0,NaN
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,0,unknown,yes,0,NaN
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,3,success,yes,1,184.0
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,0,unknown,no,0,NaN
45210,37,entrepreneur,married,secondary,no,2971,no,no,cellular,17,nov,361,2,11,other,no,1,188.0


In [122]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report

In [105]:
month_map = {
    'jan':1,
    'feb':2,
    'mar':3,
    'apr':4,
    'may':5,
    'jun':6,
    'jul':7,
    'aug':8,
    'sep':9,
    'oct':10,
    'nov':11,
    'dec':12
}
df['month_num'] = df['month'].map(month_map)
df['month_num'].value_counts()

month_num
5     13766
7      6895
8      6247
6      5341
11     3970
4      2932
2      2649
1      1403
10      738
9       579
3       477
12      214
Name: count, dtype: int64

In [106]:
df['month_sin'] = np.sin((2 * np.pi * (df['month_num'] / 12)))
df['month_cos'] = np.cos((2 * np.pi * (df['month_num'] / 12)))
df.drop('month_num', axis=1, inplace=True)

In [107]:
df['days_since_prev_contact'] = df['days_since_prev_contact'].replace(np.nan, 0)

In [108]:
x = df.drop('y', axis = 1)
y = df['y']

In [109]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [110]:
ct = ColumnTransformer([
    ('numerical', StandardScaler(), numerical_cols),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), nominal_categorical)
])

In [111]:
ct.fit(x_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [112]:
x_train_transformed = ct.transform(x_train)
x_test_transformed = ct.transform(x_test)

In [113]:
ct.get_feature_names_out()

array(['numerical__age', 'numerical__balance', 'numerical__campaign',
       'numerical__previous', 'numerical__day',
       'numerical__was_previously_contacted',
       'numerical__days_since_prev_contact', 'numerical__month_sin',
       'numerical__month_cos', 'categorical__job_admin.',
       'categorical__job_blue-collar', 'categorical__job_entrepreneur',
       'categorical__job_housemaid', 'categorical__job_management',
       'categorical__job_retired', 'categorical__job_self-employed',
       'categorical__job_services', 'categorical__job_student',
       'categorical__job_technician', 'categorical__job_unemployed',
       'categorical__job_unknown', 'categorical__marital_divorced',
       'categorical__marital_married', 'categorical__marital_single',
       'categorical__default_no', 'categorical__default_yes',
       'categorical__housing_no', 'categorical__housing_yes',
       'categorical__loan_no', 'categorical__loan_yes',
       'categorical__contact_cellular', 'categori

In [189]:
_n = 11
knn = KNeighborsClassifier(n_neighbors=_n)
knn.fit(x_train_transformed, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](2,)","['no','yes']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [190]:
y_pred = knn.predict(x_test_transformed)
y_pred[:20]

array(['no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no',
       'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no'], dtype=object)

In [191]:
y_test.iloc[:20].values

<StringArray>
[ 'no',  'no',  'no',  'no',  'no',  'no',  'no', 'yes',  'no',  'no',  'no',
  'no',  'no',  'no',  'no',  'no',  'no',  'no', 'yes',  'no']
Length: 20, dtype: str

In [192]:
m = confusion_matrix(y_test, y_pred)
data = f"""------------------- n_neighbors: {_n} --------------------\n
Accuracy: {accuracy_score(y_test, y_pred)}\n
Confusion Matrix: [{m[0, 0]}, {m[0, 1]}]
                  [{m[1, 0]}, {m[1, 1]}]\n
Classification Report:\n
{classification_report(y_test, y_pred)}
"""
print(data)

------------------- n_neighbors: 11 --------------------

Accuracy: 0.8894172287957536

Confusion Matrix: [7857, 128]
                  [872, 186]

Classification Report:

              precision    recall  f1-score   support

          no       0.90      0.98      0.94      7985
         yes       0.59      0.18      0.27      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.58      0.61      9043
weighted avg       0.86      0.89      0.86      9043




In [193]:
with open('knn_classifier_manual_optimization.txt', 'a', encoding='utf-8') as kco:
    kco.write(data)

In [217]:
#trying to construct a pipeline

#from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

In [233]:
ros_25 = RandomOverSampler(
    sampling_strategy=0.25,
    random_state=42
)

In [234]:
knn_ros25_pipeline = Pipeline([
    ('ct', ct),
    ('oversampler', ros_25),
    ('classifier', KNeighborsClassifier())
])
param_grid_ros25 = {
    'classifier__n_neighbors' : [1, 3, 5, 7, 9, 11],
    #'classifier__weights' : ['uniform', 'distance']
}
f1_yes = make_scorer(
    f1_score,
    pos_label = 'yes'
)

In [235]:
grid_search = GridSearchCV(
    estimator=knn_ros25_pipeline,
    param_grid=param_grid_ros25,
    scoring=f1_yes,
    cv=5,
    n_jobs=-1
)
grid_search.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__n_neighbors': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",make_scorer(f...pos_label=yes)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of inform

In [236]:
print('best k: ', grid_search.best_params_)
print('best f1 yes score: ', grid_search.best_score_)

best k:  {'classifier__n_neighbors': 11}
best f1 yes score:  0.371408414910286


In [237]:
best_knn_model = grid_search.best_estimator_
y_test_pred = best_knn_model.predict(x_test)

In [238]:
print("Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Accuracy: 0.8781377861329205

Confusion Matrix:
[[7578  407]
 [ 695  363]]

Classification Report:
              precision    recall  f1-score   support

          no       0.92      0.95      0.93      7985
         yes       0.47      0.34      0.40      1058

    accuracy                           0.88      9043
   macro avg       0.69      0.65      0.66      9043
weighted avg       0.86      0.88      0.87      9043



In [239]:
results = pd.DataFrame(grid_search.cv_results_)

results[
    [
        "param_classifier__n_neighbors",
        #"param_classifier__weights",
        "mean_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

,param_classifier__n_neighbors,mean_test_score,rank_test_score
5,11,0.371408,1
4,9,0.364578,2
3,7,0.351508,3
2,5,0.332529,4
1,3,0.319152,5
0,1,0.289060,6


In [242]:
best_knn_model.fit(x_train, y_train)
print(best_knn_model)

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  ['age', 'balance', 'campaign',
                                                   'previous', 'day',
                                                   'was_previously_contacted',
                                                   'days_since_prev_contact',
                                                   'month_sin', 'month_cos']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'poutcome',
                                                   'education'])])),
                ('oversampler',
                 RandomOverSa

In [245]:
import joblib
joblib.dump(
    best_knn_model,
    "bank_marketing_model.joblib"
)

['bank_marketing_model.joblib']